In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, desc, dense_rank, count, lag, lead, sum

In [0]:
sales_data = [
    (1, "Kochi", "Laptop", 50000),
    (2, "Kochi", "Mobile", 30000),
    (3, "Kochi", "Tablet", 30000),
    (4, "Kochi", "TV", 60000),
    (5, "Kochi", "Headphones", 10000),
 
    (6, "Chennai", "Laptop", 45000),
    (7, "Chennai", "Mobile", 45000),
    (8, "Chennai", "Tablet", 20000),
    (9, "Chennai", "TV", 70000),
    (10, "Chennai", "Headphones", 15000),
 
    (11, "Bangalore", "Laptop", 60000),
    (12, "Bangalore", "Mobile", 60000),
    (13, "Bangalore", "Tablet", 35000),
    (14, "Bangalore", "TV", 80000),
    (15, "Bangalore", "Headphones", 20000),
 
    (16, "Hyderabad", "Laptop", 55000),
    (17, "Hyderabad", "Mobile", 25000),
    (18, "Hyderabad", "Tablet", 25000),
    (19, "Hyderabad", "TV", 75000),
    (20, "Hyderabad", "Headphones", 12000),
 
    (21, "Delhi", "Laptop", 65000),
    (22, "Delhi", "Mobile", 40000),
    (23, "Delhi", "Tablet", 40000),
    (24, "Delhi", "TV", 85000),
    (25, "Delhi", "Headphones", 18000)
]
 
columns = ["order_id", "city", "product", "revenue"]
 
df_sales_hw = spark.createDataFrame(sales_data, columns)
df_sales_hw.show()

+--------+---------+----------+-------+
|order_id|     city|   product|revenue|
+--------+---------+----------+-------+
|       1|    Kochi|    Laptop|  50000|
|       2|    Kochi|    Mobile|  30000|
|       3|    Kochi|    Tablet|  30000|
|       4|    Kochi|        TV|  60000|
|       5|    Kochi|Headphones|  10000|
|       6|  Chennai|    Laptop|  45000|
|       7|  Chennai|    Mobile|  45000|
|       8|  Chennai|    Tablet|  20000|
|       9|  Chennai|        TV|  70000|
|      10|  Chennai|Headphones|  15000|
|      11|Bangalore|    Laptop|  60000|
|      12|Bangalore|    Mobile|  60000|
|      13|Bangalore|    Tablet|  35000|
|      14|Bangalore|        TV|  80000|
|      15|Bangalore|Headphones|  20000|
|      16|Hyderabad|    Laptop|  55000|
|      17|Hyderabad|    Mobile|  25000|
|      18|Hyderabad|    Tablet|  25000|
|      19|Hyderabad|        TV|  75000|
|      20|Hyderabad|Headphones|  12000|
+--------+---------+----------+-------+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import col
df_sales_hw.withColumn("pre_rank",lag("revenue", 1).over(Window.partitionBy("city").orderBy("order_id"))).withColumn("diff",col("revenue")-col("pre_rank")).show()

+--------+---------+----------+-------+--------+------+
|order_id|     city|   product|revenue|pre_rank|  diff|
+--------+---------+----------+-------+--------+------+
|      11|Bangalore|    Laptop|  60000|    NULL|  NULL|
|      12|Bangalore|    Mobile|  60000|   60000|     0|
|      13|Bangalore|    Tablet|  35000|   60000|-25000|
|      14|Bangalore|        TV|  80000|   35000| 45000|
|      15|Bangalore|Headphones|  20000|   80000|-60000|
|       6|  Chennai|    Laptop|  45000|    NULL|  NULL|
|       7|  Chennai|    Mobile|  45000|   45000|     0|
|       8|  Chennai|    Tablet|  20000|   45000|-25000|
|       9|  Chennai|        TV|  70000|   20000| 50000|
|      10|  Chennai|Headphones|  15000|   70000|-55000|
|      21|    Delhi|    Laptop|  65000|    NULL|  NULL|
|      22|    Delhi|    Mobile|  40000|   65000|-25000|
|      23|    Delhi|    Tablet|  40000|   40000|     0|
|      24|    Delhi|        TV|  85000|   40000| 45000|
|      25|    Delhi|Headphones|  18000|   85000|

In [0]:
df_sales_hw.withColumn("post_rank",lead("revenue").over(Window.partitionBy("city").orderBy("order_id"))).show()

+--------+---------+----------+-------+---------+
|order_id|     city|   product|revenue|post_rank|
+--------+---------+----------+-------+---------+
|      11|Bangalore|    Laptop|  60000|    60000|
|      12|Bangalore|    Mobile|  60000|    35000|
|      13|Bangalore|    Tablet|  35000|    80000|
|      14|Bangalore|        TV|  80000|    20000|
|      15|Bangalore|Headphones|  20000|     NULL|
|       6|  Chennai|    Laptop|  45000|    45000|
|       7|  Chennai|    Mobile|  45000|    20000|
|       8|  Chennai|    Tablet|  20000|    70000|
|       9|  Chennai|        TV|  70000|    15000|
|      10|  Chennai|Headphones|  15000|     NULL|
|      21|    Delhi|    Laptop|  65000|    40000|
|      22|    Delhi|    Mobile|  40000|    40000|
|      23|    Delhi|    Tablet|  40000|    85000|
|      24|    Delhi|        TV|  85000|    18000|
|      25|    Delhi|Headphones|  18000|     NULL|
|      16|Hyderabad|    Laptop|  55000|    25000|
|      17|Hyderabad|    Mobile|  25000|    25000|


In [0]:
df_sales_hw.withColumn("running_total", sum("revenue").over(Window.partitionBy("city").orderBy("order_id"))).show()


+--------+---------+----------+-------+-------------+
|order_id|     city|   product|revenue|running_total|
+--------+---------+----------+-------+-------------+
|      11|Bangalore|    Laptop|  60000|        60000|
|      12|Bangalore|    Mobile|  60000|       120000|
|      13|Bangalore|    Tablet|  35000|       155000|
|      14|Bangalore|        TV|  80000|       235000|
|      15|Bangalore|Headphones|  20000|       255000|
|       6|  Chennai|    Laptop|  45000|        45000|
|       7|  Chennai|    Mobile|  45000|        90000|
|       8|  Chennai|    Tablet|  20000|       110000|
|       9|  Chennai|        TV|  70000|       180000|
|      10|  Chennai|Headphones|  15000|       195000|
|      21|    Delhi|    Laptop|  65000|        65000|
|      22|    Delhi|    Mobile|  40000|       105000|
|      23|    Delhi|    Tablet|  40000|       145000|
|      24|    Delhi|        TV|  85000|       230000|
|      25|    Delhi|Headphones|  18000|       248000|
|      16|Hyderabad|    Lapt